# 04 - Entraînement sur GPU (Google Colab ou Kaggle)

Ce notebook permet de relancer les expériences sur GPU avec l'ensemble des 303 cas annotés au lieu
du sous-ensemble utilisé sur CPU. Il n'a pas été exécuté pour produire les résultats présentés dans
le README : ces résultats proviennent des scripts lancés en local sur CPU.

Étapes : vérification du GPU, installation, téléchargement des données, prétraitement,
entraînement des modèles (plusieurs seeds), évaluation, export des résultats.

Sur Colab : Exécution > Modifier le type d'exécution > GPU. Sur Kaggle : activer l'accélérateur GPU
et l'accès Internet dans les paramètres du notebook.

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| CUDA :", torch.cuda.is_available())

## 1. Récupération du projet

Le dépôt doit être accessible (public, ou cloné avec un jeton personnel saisi par l'utilisatrice).
On peut aussi téléverser une archive .zip du projet et la décompresser.

In [ ]:
REPO = "https://github.com/HaoudarNouhaila/wavelet-hepatic-vessel-segmentation.git"
!git clone -q $REPO project
%cd project

## 2. Installation

PyTorch, NumPy, pandas et Matplotlib sont déjà installés sur Colab et Kaggle ; seules les
bibliothèques manquantes sont ajoutées.

In [ ]:
!pip install -q monai==1.6.1 PyWavelets==1.8.0 nibabel==5.4.2 SimpleITK==2.5.6 scikit-image==0.25.2
!python scripts/check_environment.py

## 3. Données

Sur Colab ou Kaggle, le débit permet de télécharger l'archive complète (9,3 Go) et d'extraire
directement imagesTr et labelsTr. La séparation au niveau patient est ensuite recréée à
l'identique (seed 42, 70 / 15 / 15) à partir des fichiers locaux, et le sous-ensemble est étendu à
tous les cas de chaque ensemble : 212 en entraînement, 45 en validation, 46 en test.

In [ ]:
!mkdir -p data/raw
!curl -s https://msd-for-monai.s3-us-west-2.amazonaws.com/Task08_HepaticVessel.tar | tar -x -C data/raw --exclude="*/imagesTs/*" --exclude="*/._*"
!ls data/raw/Task08_HepaticVessel/imagesTr | wc -l

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/data.yaml"))
cfg["split"]["subset_sizes"] = {"train": 212, "val": 45, "test": 46}
yaml.safe_dump(cfg, open("configs/data_full.yaml", "w"), sort_keys=False, allow_unicode=True)
!python scripts/download_data.py --config configs/data_full.yaml --from-local
!python scripts/prepare_data.py --config configs/data_full.yaml
!python scripts/inspect_dataset.py --config configs/data_full.yaml

## 4. Entraînement

Mêmes configurations que sur CPU, avec plus de patches par époque. Les trois modèles 2D voient les
mêmes patches pour une même seed. Pour garder la comparaison équitable, ne modifier les paramètres
que de la même façon pour tous les modèles.

In [ ]:
SEEDS = [0, 1, 2]
CONFIGS = ["configs/unet_2d.yaml", "configs/wavelet_unet.yaml", "configs/wavelet_unet_multiscale.yaml"]
for seed in SEEDS:
    for config in CONFIGS:
        !python scripts/train.py --config {config} --seed {seed} --samples-per-epoch 2000

In [ ]:
# Extension 3D (facultative)
!python scripts/train.py --config configs/unet_3d.yaml --seed 0

## 5. Évaluation et analyse

L'ensemble de test n'est utilisé qu'ici. La liste des expériences évaluées est écrite dans une
configuration dédiée.

In [ ]:
from pathlib import Path
ev = yaml.safe_load(open("configs/evaluation.yaml"))
ev["data_config"] = "configs/data_full.yaml"
ev["runs"] = sorted(str(p.parent) for p in Path("experiments").glob("*_seed*/best.pt"))
yaml.safe_dump(ev, open("configs/evaluation_full.yaml", "w"), sort_keys=False, allow_unicode=True)
print(ev["runs"])
!python scripts/evaluate.py --config configs/evaluation_full.yaml
!python scripts/analyze_results.py --split test --eval-config configs/evaluation_full.yaml

## 6. Export

Les checkpoints et les prédictions ne sont pas versionnés dans Git : ils sont regroupés dans une
archive à télécharger (ou à copier sur Google Drive après `drive.mount`, qui demande une
authentification).

In [ ]:
!zip -q -r resultats_gpu.zip results experiments -x "experiments/predictions/*"
from pathlib import Path
print(Path("resultats_gpu.zip").stat().st_size / 2**20, "Mo")